In [1]:
import sys
from pathlib import Path
try:
    current_dir = Path(__file__).parent
except NameError:
    current_dir = Path.cwd()
root_dir = current_dir.parent.parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="build")

# Iterate over the attributes of the DirPaths object
for attr in dir(dirs):
    if attr.startswith('_'):
        continue
    if callable(getattr(dirs, attr)):
        continue
    print(f"{attr}: {getattr(dirs, attr)}")

data_dir: D:\FAME - LN dataset\Dropbox\fame_clean\2_clean_FAME_data
db_path: C:\Users\lazyst\Files\ucl\Dissertation\build\output\fame_data.duckdb
input_dir: C:\Users\lazyst\Files\ucl\Dissertation\build\input
output_dir: C:\Users\lazyst\Files\ucl\Dissertation\build\output
raw_data_dir: D:\FAME - LN dataset\Dropbox\fame_clean\1_FAME_raw_data\2025.02
root_data_dir: D:\FAME - LN dataset\Dropbox\fame_clean
root_dir: C:\Users\lazyst\Files\ucl\Dissertation
tmp_dir: C:\Users\lazyst\Files\ucl\Dissertation\build\tmp
work_dir: C:\Users\lazyst\Files\ucl\Dissertation\build\src


# 1. Ingest from FAME

### Categorise raw files

This script resolves the project data paths, scans the raw-data folder, builds a nested dictionary of file metadata by company and file category, and writes it to JSON.  
`get_data_dirs(segment="build")` defines the working directories  
`build_raw_file_dict()` performs the recursive traversal and file collection through its helper functions.  

In [2]:
from flask import json
import pandas as pd
from utils.f_0_dirs import get_data_dirs
from f_1_traverse import build_raw_file_dict

def convert(seconds):
    seconds = seconds % (24 * 3600)
    hour = seconds // 3600
    seconds %= 3600
    minutes = seconds // 60
    seconds %= 60
    return "%dh:%02dm:%02ds" % (hour, minutes, seconds)

rate = 1 # 1 second per file
dirs = get_data_dirs(segment="build")

pd.options.mode.chained_assignment = None 

if dirs.raw_data_dir is None:
    raise ValueError("raw_data_dir is None. Please check your .env file and ensure RAW_DATA_DIR is set correctly.")

raw_file_dict, file_count, processed = build_raw_file_dict(dirs.raw_data_dir)

# Save dict
with open(dirs.output_dir / "raw_file_dict.json", "w") as f:
    json.dump(raw_file_dict, f, indent=4)
print(f"✅ Successfully built raw file dictionary and saved to: {dirs.output_dir / 'raw_file_dict.json'}")
print(f"✅ Processed {processed:,} file paths.")

# Build File Count DataFrame
file_count_df = pd.DataFrame(file_count).T.reset_index().rename(columns={'index': 'industry'})
file_count_df['total'] = file_count_df.drop(columns=['industry']).sum(axis=1)
with open(dirs.root_dir / "build" / "tmp" / "file_count.md", "w") as f:
    f.write(file_count_df.to_markdown(index=False))
priority_path = dirs.root_dir / "build" / "input" / "SIC_priorities.json"
sic_priorities = []
with open(priority_path, "r") as f:
    sic_priorities = json.load(f)
for item in sic_priorities:
    division = item["Division"]
    if division in file_count_df['industry'].values:
        counts = file_count_df[file_count_df['industry'] == division].iloc[0].to_dict()
        item["Files Count"] = {k: v for k, v in counts.items() if k != 'industry'}
with open(priority_path, "w") as f:
    json.dump(sic_priorities, f, indent=4)
print(f"✅ Successfully built file count markdown table.")

# Get Highest Totals
ind_max_row = file_count_df.loc[file_count_df['total'].idxmax()]

# Unpivot, excluding 'total' beforehand to save time processing it
melted = file_count_df.drop(columns=['total']).melt(id_vars=['industry'], var_name='property', value_name='value')
prop_max_row = melted.loc[melted['value'].idxmax()]

# Stats
peak_ram_required = prop_max_row['value'] * 6 / 1024 * 2 # in GB

print(f"⚠️ Max industry files: industry '{ind_max_row['industry']}' with {ind_max_row['total']} files.")
print(f"⚠️ Max property files: industry '{prop_max_row['industry']}'/property '{prop_max_row['property']}' with {prop_max_row['value']} files.")
print(f"⚠️ Estimated peak RAM required: {peak_ram_required:.2f} GB (for industry '{prop_max_row['industry']}' and property '{prop_max_row['property']}')")
print(f"⏱️ Estimated time to process all files: {convert(processed * rate)} (at {rate} second per file)")

Traversing industry directory: 01, 02, 03, 05, 06, 07, 08, 09, 10, 11, 12, 13, 14, 15, 16
17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31
32, 33, 35, 36, 37, 38, 39, 41, 42, 43, 45, 46, 47, 49, 50
51, 52, 53, 55, 56, 58, 59, 60, 61, 62, 63, 64, 65, 66, 68
69, 70, 71, 72, 73, 74, 75, 77, 78, 79, 80, 81, 82, 84, 85
86, 87, 88, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99
✅ Successfully built raw file dictionary and saved to: C:\Users\lazyst\Files\ucl\Dissertation\build\output\raw_file_dict.json
✅ Processed 12,846 file paths.
✅ Successfully built file count markdown table.
⚠️ Max industry files: industry '47' with 1090 files.
⚠️ Max property files: industry '47'/property 'a4_profits' with 431 files.
⚠️ Estimated peak RAM required: 5.05 GB (for industry '47' and property 'a4_profits')
⏱️ Estimated time to process all files: 3h:34m:06s (at 1 second per file)


### [read] Load in the master schema

In [3]:
from typing import TypedDict
import pandas as pd

# import xlsx file from input/raw_properties.xlsx to load as a schema
# Declare types that the schema_source df has colums ["from_raw", "key", "type", "fuzzy_mapping", "in_raw_data", "keep", "in_ln_set", "description"]
schema_path = dirs.input_dir / "raw_properties.xlsx"

# cast types
# columns 'in_ln_set' and 'may_mix' are boolean, but some rows are nan
# cast nan to false and ensure the column is a proper boolean
schema_source = pd.read_excel(schema_path, sheet_name="raw_properties", engine="calamine",
    dtype={
        "from_raw": str,
        "key": str,
        "type": str,
        "fuzzy_mapping": str,
        "keep": str,
        "in_ln_set": "boolean",
        "may_mix": "boolean",
        "deflate": str,
        "description": str
    }
)
# Iterate over columns where type is boolean and fillna with False
for col in schema_source.select_dtypes(include='boolean').columns:
    schema_source[col] = schema_source[col].fillna(False)
schema_raw: pd.DataFrame = schema_source[schema_source["from_raw"].notna()]

# Take our master schema and turn it into a helpful mapping of fuzzy column names to schema column names
# Regardless of what the raw source is. We'll filter it later
class SRFbyProperty(TypedDict):
    mapping: pd.DataFrame
    col_map: dict[str, str]

properties = ['a1_ID', 'a2_key_finance', 'a3_assets', 'a4_profits', 'a5_misc']
start_year = 2006
end_year = 2025
schema_fixed_fuzzy_df: pd.DataFrame = schema_source[["key", "fuzzy_mapping"]]
schema_fixed_fuzzy_by_property = {}
schema_yearly_cols_by_property = {}

for p in properties:
    srf_filtered: pd.DataFrame = schema_raw[schema_raw["from_raw"].isin([p, 'all'])]
    srf_filtered["fml"] = srf_filtered["fuzzy_mapping"].str.split('\n')
    # Just make 
    srf_mapping: pd.DataFrame = srf_filtered
    # For every value in srf_filtered["fml"]
    # Create a dict which is that value, and the corresponding entry in srf_filtered["key"]
    # But only if the value is not NaN
    srf_col_map = {
        fml: key
        for _, row in srf_filtered.iterrows()
        if row["fml"] is not None and isinstance(row["fml"], list)
        for fml in row["fml"]
        for key in [row["key"]]
    }
    exact_cols = schema_raw[(schema_raw["from_raw"].isin([p, 'all'])) & (schema_raw["keep"] != "yearly")]["key"].tolist()
    yearly_cols = schema_raw[(schema_raw["from_raw"].isin([p, 'all'])) & (schema_raw["keep"] == "yearly")]["key"].tolist()
    for year in range(start_year, end_year + 1):
        for col in yearly_cols:
            # Get the fml for this col
            fmls = srf_filtered[srf_filtered["key"] == col]["fml"].values[0]
            if fmls is None or not isinstance(fmls, list):
                continue
            for fml in fmls:
                srf_col_map[f"{fml}\n{year}"] = f"{col}@{year}"

    
    # Create a dataframe called yearly_df_all_possible_years which is identical to srf_mapping
    # but for every yearly_col, it has a new row for every year between start_year and end_year
    # but the key has the @year suffix
    # Create this dataframe then concat it to srf_mapping
    schema_raw_p_all_possible_years = pd.DataFrame()
    for year in range(start_year, end_year + 1):
        yearly_df = srf_mapping[srf_mapping["key"].isin(yearly_cols)].copy()
        yearly_df["key"] = yearly_df["key"].apply(lambda x: f"{x}@{year}")
        yearly_df["fuzzy_mapping"] = yearly_df["fuzzy_mapping"].apply(lambda x: f"{x}\n{year}" if isinstance(x, str) else x)
        yearly_df["fml"] = yearly_df["fml"].apply(lambda x: [f"{fml}\n{year}" for fml in x] if isinstance(x, list) else x)
        schema_raw_p_all_possible_years = pd.concat([schema_raw_p_all_possible_years, yearly_df], ignore_index=True)

    schema_fixed_fuzzy_by_property[p] = {
        "mapping": srf_mapping,
        "col_map": srf_col_map
    }

    # Handle yearly properties
    schema_yearly_cols_by_property[p] = {
        "exact": exact_cols,
        "yearly": yearly_cols,
        "df": schema_raw_p_all_possible_years
    }
schema_raw_all_possible_years = pd.concat([schema_raw] + [schema_yearly_cols_by_property[p]["df"] for p in properties], ignore_index=True)
# Output this schema to a json in the build / tmp file
with open(dirs.root_dir / "build" / "tmp" / "schema_raw_all_possible_years.json", "w") as f:
    schema_raw_all_possible_years.to_json(f, orient="records", indent=4)
    print(f"✅ Successfully built schema_raw_all_possible_years and saved to: {dirs.root_dir / 'build' / 'tmp' / 'schema_raw_all_possible_years.json'}")

# Print any rows where fml has more than one element
for p in properties:
    mapping = schema_fixed_fuzzy_by_property[p]["mapping"]

# Export the 5 col_maps to a pretty json in tmp / fuzzy_col_mapping.json
with open(dirs.root_dir / "build" / "tmp" / "fuzzy_col_mapping.json", "w") as f:
    json.dump({p: schema_fixed_fuzzy_by_property[p]["col_map"] for p in properties}, f, indent=4)
    print(f"✅ Successfully built fuzzy column mapping and saved to: {dirs.root_dir / 'build' / 'tmp' / 'fuzzy_col_mapping.json'}")

✅ Successfully built schema_raw_all_possible_years and saved to: C:\Users\lazyst\Files\ucl\Dissertation\build\tmp\schema_raw_all_possible_years.json
✅ Successfully built fuzzy column mapping and saved to: C:\Users\lazyst\Files\ucl\Dissertation\build\tmp\fuzzy_col_mapping.json


# 2. Modify and output to duckdb

### [read/write] Define duck schemas and reset tables if non-matching

In [4]:
import pandas as pd
import ibis
# pip install 'ibis-framework[duckdb,geospatial]'

force_overwrite = False

print("Path:", ibis.__file__)
print("Version:", getattr(ibis, "__version__", "No version found"))
pd.options.mode.chained_assignment = None  # default='warn'

# Fixed schema
# schema_fixed is a df of schema_source where values in column "keep" are "fixed" or "all"
schema_fixed: pd.DataFrame = schema_source[schema_source["keep"].isin(["fixed", "all"])]
schema_fixed_dict: dict[str, str] = dict(zip(schema_fixed["key"], schema_fixed["type"]))
schema_fixed_ibis: ibis.Schema = ibis.schema(schema_fixed_dict)
schema_fixed_names: set[str] = set(schema_fixed_ibis.keys())

schema_derived: pd.DataFrame = schema_source[schema_source["keep"].isin(["derived", "all"])]
schema_derived_dict: dict[str, str] = dict(zip(schema_derived["key"], schema_derived["type"]))
schema_derived_ibis: ibis.Schema = ibis.schema(schema_derived_dict)
schema_derived_names: set[str] = set(schema_derived_ibis.keys())

# build a schema_yearly df with columns registered_number, fame_key, year, value properties, with types str, str, int, float
schema_yearly: pd.DataFrame = pd.DataFrame({
    "key": ["registered_number", "fame_key", "year", "value"],
    "type": ["string", "string", "int64", "float64"]
})
schema_yearly: pd.DataFrame = schema_source[schema_source["keep"].isin(["yearly", "all"])]
schema_yearly_dict: dict[str, str] = dict(zip(schema_yearly["key"], schema_yearly["type"]))
schema_yearly_ibis: ibis.Schema = ibis.schema(schema_yearly_dict)
schema_yearly_names: set[str] = set(schema_yearly_ibis.keys())

# 4. Execute the table creation using the Ibis schema
con = ibis.duckdb.connect(str(dirs.db_path))
def create_or_update_table(table_name: str, target_schema: ibis.Schema):
    print(f"Initializing DuckDB via Ibis at: {dirs.db_path}")
    try:
        existing_tables = con.list_tables()
        if table_name not in existing_tables or force_overwrite:
            # Table doesn't exist at all -> Create it
            con.create_table(table_name, schema=target_schema)
            print(f"✅ Successfully created Ibis schema for '{table_name}'.")
            
        else:
            # Table exists -> Fetch its current schema
            current_schema = con.table(table_name).schema()
            
            # Compare the existing schema to our target schema
            if current_schema == target_schema:
                print(f"⏩ Table '{table_name}' exists and schema matches perfectly. Skipping.")
            else:
                print(f"⚠️ Table '{table_name}' schema mismatch detected! Overwriting...")
                # overwrite=True safely drops the old table and creates the new one
                con.create_table(table_name, schema=target_schema, overwrite=True)
                print(f"✅ Successfully overwrote '{table_name}' with updated schema.")

        # Print verification
        row_count = con.table(table_name).count().execute()
        print(f"--- Schema Verification ({table_name}):")
        print(f"--- Columns ({table_name}): {", ".join(list(con.table(table_name).schema().names))}")
        print(f"--- Row Count ({table_name}): {row_count}")

    except Exception as e:
        print(f"❌ Error creating table: {e}")


# Execute for all three tables
create_or_update_table("fame_fixed", schema_fixed_ibis)
create_or_update_table("fame_derived", schema_derived_ibis)
create_or_update_table("fame_yearly", schema_yearly_ibis)

Path: c:\Users\lazyst\AppData\Local\python\pythoncore-3.14-64\Lib\site-packages\ibis\__init__.py
Version: 12.0.0
Initializing DuckDB via Ibis at: C:\Users\lazyst\Files\ucl\Dissertation\build\output\fame_data.duckdb
⏩ Table 'fame_fixed' exists and schema matches perfectly. Skipping.
--- Schema Verification (fame_fixed):
--- Columns (fame_fixed): company_name, registered_number, ticker_symbol, ro_address, ro_address_line_1, ro_address_line_2, ro_address_line_3, ro_address_line_4, ro_address_line_5, ro_city, ro_county, ro_postcode, ro_full_postcode, ro_country, ro_latitude, ro_longitude, ro_nuts_region, ro_postal_region, primary_trading_address, primary_trading_address_latitude, primary_trading_address_longitude, branch_name, primary_uk_sic_2007_code, primary_uk_sic_2007_description, latest_accounts_date, no_of_available_years, guo, guo_nb, entity_type
--- Row Count (fame_fixed): 3700657
Initializing DuckDB via Ibis at: C:\Users\lazyst\Files\ucl\Dissertation\build\output\fame_data.duckdb


### Load, modify and write imported file schema

In [ ]:
import traceback
from concurrent.futures import ProcessPoolExecutor
from flask import json
from f_1_traverse import RawFileDict
from f_2_workers import ingest_single_excel_file
from f_2_check import check_df_matches_schema, handle_excel_dates, handle_mixed_types, rename_df_with_years
from f_2_modify import coerce_ibis_dates_from_schema, reindex_ibis_table
import random

test_flag = False
ingest_industries = False # ['47']

skip_industries = ['01', '02', '03', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '35', '36', '37', '38', '39', '41', '42', '45', '46', '49', '50', '51', '52', '53', '55', '56', '58', '59', '60', '61', '62', '63', '64', '65', '66', '68', '69', '70', '71', '72', '73', '75', '77', '78', '79', '80', '81', '82', '84', '85', '86', '87', '88', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99']

# Traverse the raw_file_dict.json file to get each Excel filepath
# declare raw_file_dict as a RawFileDict type
raw_file_dict: RawFileDict | None = None
with open(dirs.output_dir / "raw_file_dict.json", "r") as f:
    raw_file_dict = json.load(f)
if raw_file_dict is None:
    raise ValueError("❌ Error: raw_file_dict.json is empty or not found.")
if dirs.raw_data_dir is None:
    raise ValueError("❌ Error: raw_data_dir is None. Please check your .env file and ensure RAW_DATA_DIR is set correctly.")

# Get my industry list
# Open build / input / SIC_priorities.json and get the list of industries in order of priority

ind_keys = list(raw_file_dict.keys())
if test_flag:
    random.shuffle(ind_keys)
elif isinstance(ingest_industries, list) and len(ingest_industries) > 0:
    ind_keys = [ind for ind in ind_keys if ind in ingest_industries]
elif isinstance(skip_industries, list) and len(skip_industries) > 0:
    ind_keys = [ind for ind in ind_keys if ind not in skip_industries]
elif ingest_industries:
    with open(dirs.root_dir / "build" / "input" / "SIC_priorities.json", "r") as f:
        sic_priorities = json.load(f)
        ind_keys = [item["Division"] for item in sic_priorities if item["Division"] in raw_file_dict]
random.shuffle(ind_keys)

ind_count = 0
process_count = 0
time_start = pd.Timestamp.now()
for ind in ind_keys:
    obj = raw_file_dict[ind]

    print(f"Ingesting industry: {ind} with {len(obj)} properties.")
    if test_flag and process_count > 200:
        break

    batches_raw: dict[str, list[pd.DataFrame]] = { p: [] for p in properties }
    ind_batches_masters: dict[str, pd.DataFrame] = { p: pd.DataFrame() for p in properties }

    ind_count += 1
    if test_flag and ind_count > 3:
        break
    
    for property, arr in obj.items():
        if not arr:
            print(f"⚠️ Warning: No files found for property '{property}' in industry '{ind}'. Skipping.")
            continue

        time_end = pd.Timestamp.now()
        time_taken = (time_end - time_start).total_seconds()
        time_rate = time_taken / process_count if process_count > 0 else 0
  
        files_shuffled = arr.copy()
        if test_flag and len(files_shuffled) > 3:
            files_shuffled = files_shuffled[:3]

        additional_info = f" (test: {len(files_shuffled)} files)" if test_flag else ""
        print(f"--- Ingesting property: {property} with {len(arr)} files{additional_info}. Rate: {time_rate:.1f} seconds/file")
        property_batch: list[pd.DataFrame] = []
        
        # Extract variables needed for the workers
        schema_raw_fuzzy_col_map = schema_fixed_fuzzy_by_property[property]['col_map']
        exact_cols = schema_yearly_cols_by_property[property]["exact"]
        yearly_cols = schema_yearly_cols_by_property[property]["yearly"]

        random.shuffle(files_shuffled)
        process_count += len(files_shuffled)

        # 1. Package the arguments for the workers
        worker_args = [
            (file_name, file_path, property, schema_raw_fuzzy_col_map, start_year, end_year, ind, exact_cols, yearly_cols)
            for file_name, file_path in files_shuffled
        ]

        # 2. Execute the file reading in parallel! Capped at 4 workers for 8GB RAM safety.
        with ProcessPoolExecutor(max_workers=4) as executor:
            # Map the arguments to the worker function
            # This worker function reads the Excel file, and returns a df with normalised columns, and filtered
            results = executor.map(ingest_single_excel_file, worker_args)
            
            for df_result in results:
                if df_result is None:
                    print(f"⚠️ Warning: A file failed to process for property '{property}' in industry '{ind}'.")
                else:
                    property_batch.append(df_result)

        if not property_batch:
            print(f"⚠️ Warning: All files failed to process for property '{property}'. Skipping.")
            continue

        # ==========================================
        # OUTSIDE THE LOOP: Vectorized Batch Cleaning
        # ==========================================
        df_raw_batch = pd.concat(property_batch, ignore_index=True)
        del property_batch

        # 3. VECTORIZED REGEX (Instantaneous on the whole block)
        df_raw_batch['registered_number'] = df_raw_batch['registered_number'].astype(str).str.replace(r'\.0$', '', regex=True)
        df_raw_batch = df_raw_batch[~df_raw_batch['registered_number'].isin(['nan', 'None', '', '<NA>'])]

        df_raw_batch2 = handle_mixed_types(schema_raw_all_possible_years, df_raw_batch, ref=f"{ind}/{property}")
        df_raw_batch2 = handle_excel_dates(df_raw_batch2)
        df_raw_batch3 = df_raw_batch2.copy()
        df_raw_batch3['industry_codes'] = ind

        ind_batches_masters[property] = df_raw_batch3.drop_duplicates(
            subset=['registered_number'], keep='first'
        ).copy()
        del df_raw_batch3

    batch_has_data = [p for p in properties if ind_batches_masters[p] is not None and not ind_batches_masters[p].empty]
    print(f"Deriving ingested files for industry: {ind}. Properties with data: {', '.join(batch_has_data)}")
    
    try:

        # FILTER, processing the a1_ID table first.
        if len(ind_batches_masters["a1_ID"]) > 0:
            schema_raw_fuzzy_mapping = schema_fixed_fuzzy_by_property["a1_ID"]["mapping"]
            df_id = ind_batches_masters["a1_ID"]
            if 'no_of_available_years' in df_id.columns:
                df_id = df_id[df_id['no_of_available_years'] != 0]
            if 'ro_country' in df_id.columns:
                df_id = df_id[df_id['ro_country'] != "Republic of Ireland"]
            ind_batches_masters["a1_ID"] = df_id

            try:
                check_df_matches_schema(schema_raw_fuzzy_mapping, ind_batches_masters["a1_ID"])           # Check if the DataFrame matches the input
            except ValueError as e:
                print(f"⚠️ Mismatch in raw data validation for file {ind}/a1_ID)")
                print("Warning:", e)


        # MODIFY for DERIVED table
        table_t_id = ibis.memtable(ind_batches_masters["a1_ID"] if ind_batches_masters["a1_ID"] is not None else pd.DataFrame())
        table_t_misc = ibis.memtable(ind_batches_masters["a5_misc"] if ind_batches_masters["a5_misc"] is not None else pd.DataFrame())
        table_t_merged_id_misc = table_t_id.left_join(table_t_misc, "registered_number").select(
            *[table_t_id[col] for col in table_t_id.columns],
            *[table_t_misc[col] for col in table_t_misc.columns if col not in table_t_id.columns]
        )
        table_t_fixed = reindex_ibis_table(schema_fixed_ibis, table_t_merged_id_misc)
        table_t_fixed_dates = coerce_ibis_dates_from_schema(schema_fixed_ibis, table_t_fixed)
        table_fixed_type_casts = {
            col: table_t_fixed_dates[col].try_cast(schema_fixed_ibis.fields[col])
            for col in schema_fixed_names if col in table_t_fixed_dates.columns
        }
        table_t_fixed_cast = table_t_fixed_dates.mutate(**table_fixed_type_casts).select(schema_fixed_names)
        con.insert("fame_fixed", table_t_fixed_cast)
        print(f"✅ Successfully appended fixed table from: {ind}")
            
        # Safely extract columns, defaulting to an Ibis null object if missing from raw data
        col_pta = table_t_id['primary_trading_address'] if 'primary_trading_address' in table_t_id.columns else ibis.null()
        col_lat = table_t_id['primary_trading_address_latitude'] if 'primary_trading_address_latitude' in table_t_id.columns else ibis.null()
        col_lon = table_t_id['primary_trading_address_longitude'] if 'primary_trading_address_longitude' in table_t_id.columns else ibis.null()
        col_ticker = table_t_id['ticker_symbol'] if 'ticker_symbol' in table_t_id.columns else ibis.null()
        col_comp = table_t_id['company_name'] if 'company_name' in table_t_id.columns else ibis.null()
        col_branch = table_t_id['branch_name'] if 'branch_name' in table_t_id.columns else ibis.null()
        col_ind = table_t_id['industry_codes'] if 'industry_codes' in table_t_id.columns else ibis.null()
        col_file = table_t_id['file_codes'] if 'file_codes' in table_t_id.columns else ibis.null()

        table_derived = table_t_id.mutate(
            has_ptaddress = col_pta.notnull(),
            has_ptaddress_latlong = col_lat.notnull() & col_lon.notnull(),
            is_public = col_ticker.notnull(),
            has_company_branch_mismatch = col_comp != col_branch,
            industry_codes = col_ind,
            file_codes = col_file
        ).select(schema_derived_names)

        con.insert("fame_derived", table_derived)
        print(f"✅ Successfully appended derived table from: {ind}")


        # YEARLY VARIABLES processing
        print(f"Processing yearly variables for industry: {ind}.")
        yearly_tables = []             # 3 dataframes
        for p in ["a2_key_finance", "a3_assets", "a4_profits"]:
            if p not in ind_batches_masters or ind_batches_masters[p].empty:
                continue

            # WIDE: 3 raw tables each with consolidated@2026 etc variables
            # SKINNY: same as wide but cut out any excess columns we might have had - drop 'industry_codes', 'file_codes'
            # Pivot to long casting to string valuables to use as universal transport. We cast back later.
            df_wide = ind_batches_masters[p]
            yearly_cols = [col for col in df_wide.columns if '@' in col]
            cols_to_keep = ["registered_number"] + yearly_cols
            df_skinny = df_wide[cols_to_keep].copy()
            del df_wide
            table_t_skinny = ibis.memtable(df_skinny)
            
            cast_dict = {col: table_t_skinny[col].cast('string') for col in yearly_cols}
            table_t_skinny_uniform = table_t_skinny.mutate(**cast_dict)

            table_t_long = table_t_skinny_uniform.pivot_longer(
                ibis.selectors.contains('@'),
                names_to="raw_key",
                values_to="value"
            ).filter(ibis._["value"].notnull())  
            yearly_tables.append(table_t_long)

        if len(yearly_tables) == 0:
            print(f"⚠️ Warning: No yearly data found for industry '{ind}'. Skipping.")
            continue

        # LONG: extract the year from the key column, so we have two keys: variable and year.
        table_t_yearly_long = ibis.union(*yearly_tables)
        table_split = table_t_yearly_long.mutate(
            base_variable = table_t_yearly_long['raw_key'].split('@')[0], # type: ignore
            year = table_t_yearly_long['raw_key'].split('@')[1].cast('int32') # type: ignore
        ).drop('raw_key')

        # PANEL: pivot_wider back to wider table, now that we have year separately, so we can have each of consolidated, turnover, etc. Per year.
        table_t_yearly = table_split.pivot_wider(
            names_from="base_variable",
            values_from="value"
        )
        # Ensure that table_t_yearly exactly only matches columns in schema
        table_t_yearly_selected = table_t_yearly.mutate(
            **{col: ibis.null() for col in schema_yearly_names if col not in table_t_yearly.columns}
        ).select(schema_yearly_names)

        con.insert("fame_yearly", table_t_yearly_selected)
        print(f"✅ Successfully appended unpivoted yearly variables from: {ind}")

    except Exception as e:
        print(f"❌ Error processing folder {ind}")
        print(f"❌ Pipeline failed: {type(e).__name__} - {e}")
        traceback.print_exc() # This prints the full red error log so you know the exact line

Ingesting industry: 29 with 5 properties.
--- Ingesting property: a1_ID with 1 files. Rate: 0.0 seconds/file
--- Ingesting property: a2_key_finance with 2 files. Rate: 2.2 seconds/file
--- Ingesting property: a3_assets with 6 files. Rate: 1.2 seconds/file
--- Ingesting property: a4_profits with 6 files. Rate: 0.7 seconds/file
--- Ingesting property: a5_misc with 1 files. Rate: 0.6 seconds/file
Deriving ingested files for industry: 29. Properties with data: a1_ID, a2_key_finance, a3_assets, a4_profits, a5_misc
✅ Successfully appended fixed table from: 29
✅ Successfully appended derived table from: 29
Processing yearly variables for industry: 29.
✅ Successfully appended unpivoted yearly variables from: 29
Ingesting industry: 14 with 5 properties.
--- Ingesting property: a1_ID with 2 files. Rate: 4.3 seconds/file
--- Ingesting property: a2_key_finance with 5 files. Rate: 3.9 seconds/file
--- Ingesting property: a3_assets with 13 files. Rate: 3.1 seconds/file
--- Ingesting property: a4_pro


### [read xlsx/append duckdb] Add Lars data tables

- Create new tables for lars_fixed and lars_yearly.
- Schema lars_yearly: `registered_number,company_name,year,consolidated,employees,systemA_industry,cons_uncons,turnover_th_gbp,shareholders_funds_th_gbp,profit_loss_before_taxation_th_gbp,number_of_employees,total_assets_th_gbp,ebitda_th_gbp,current_liabilities_th_gbp,long_term_liabilities_th_gbp,research_development_th_gbp,wages_salaries_th_gbp,interest_paid_th_gbp,taxation_th_gbp,depreciation_th_gbp,tangible_assets_th_gbp,intangible_assets_th_gbp,fixed_assets_th_gbp,other_fixed_assets_th_gbp,pension_costs_th_gbp,social_security_costs_th_gbp,dividends_distributable_profit_th_gbp,systemB_group,cost_of_sales_th_gbp,exceptional_items_pre_gp_th_gbp,cash_out_in_flow_investing_activ_th_gbp,capital_expenditure_financ_invest_th_gbp,acquisition_disposal_th_gbp,equity_dividends_paid_th_gbp,company_name_raw`
which is `str`, `str`, then float for the rest
- Schema lars_fixed: `registered_number,company_name_A,primary_uk_sic_2007_code,primary_uk_sic_2007_description,full_overview,primary_business_line,no_of_available_years,latest_accounts_date,company_name_B,inactive,quoted,own_data,woco,bv_d_id_number,company_status,status_date,legal_form,date_of_incorporation,accounting_reference_date,registered_accounts_type,jordans_company_classification,account_currency,guo_name,guo_bv_d_id_number,duo_name,duo_bv_d_id_number`
autodetect types
- Also, lars data has a registered_number that sometimes has a hash (#) in front of its 7-digits
which we should manipulate to match our above
- Otherwise, this should nicely match our existing tables and we can match up rows later

In [11]:

from utils.f_0_dirs import get_data_dirs
dirs = get_data_dirs(segment="build")

try:
    if dirs.data_dir is None:
        raise ValueError("❌ Error: data_dir is None. Please check your .env file and ensure DATA_DIR is set correctly.")
    
    lars_fixed_path = dirs.data_dir / "firms_final_sample_fixed_vars.csv"
    lars_yearly_path = dirs.data_dir / "firms_final_sample_yearly_vars.csv"

    df_lars_fixed = pd.read_csv(lars_fixed_path, dtype=str)
    df_lars_yearly = pd.read_csv(lars_yearly_path, dtype=str)

    # Clean up registered_number in both dataframes
    df_lars_fixed['registered_number'] = df_lars_fixed['registered_number'].str.replace(r'^#', '', regex=True)
    df_lars_yearly['registered_number'] = df_lars_yearly['registered_number'].str.replace(r'^#', '', regex=True)

    # Insert into DuckDB
    con.create_table("lars_fixed", schema=ibis.schema({col: 'string' for col in df_lars_fixed.columns}), overwrite=True)
    con.insert("lars_fixed", df_lars_fixed)

    con.create_table("lars_yearly", schema=ibis.schema({col: 'string' for col in df_lars_yearly.columns}), overwrite=True)
    con.insert("lars_yearly", df_lars_yearly)

    print("✅ Successfully ingested Lars data tables.")
except Exception as e:
    print(f"❌ Error ingesting Lars data tables: {e}")
    traceback.print_exc()

✅ Successfully ingested Lars data tables.


### [read] basic tables overview
- List tables in the DuckDB database as an .md file in /tmp
- Give me the head of all tables
- Ensure they are nicely formatted with headers so I can easily see what's going on

In [10]:
import ibis
from utils.f_0_dirs import get_data_dirs
dirs = get_data_dirs(segment="build")

out_file = dirs.output_dir / "duckdb_tables.md"
con = ibis.duckdb.connect(str(dirs.output_dir / "fame_data.duckdb"))
interesting_tables = ["fame_fixed", "fame_derived", "fame_yearly", "lars_fixed", "lars_yearly"]
existing_tables = con.list_tables()
intersection = set(interesting_tables).intersection(existing_tables)

with open(out_file, "w") as f:
    f.write("# Tables in DuckDB database\n\n")
    for table in intersection:
        f.write(f"## {table}\n\n")
        f.write(f"### Number of rows: {con.table(table).count().execute():,}\n\n")
        f.write(f"### Schema:\n\n```\n{con.table(table).schema()}\n```\n\n")
        f.write(f"### Head of table:\n\n```\n{con.table(table).head().execute()}\n```\n\n")
        if table == "fame_derived":
            # Fetch all the unique values in the columns industry_codes
            industry_codes = con.table(table).select("industry_codes").distinct().execute()
            industry_codes_sorted = sorted(industry_codes["industry_codes"].dropna().unique())
            f.write(f"### Unique industry_codes in {table}: {", ".join(industry_codes_sorted)}")
print(f"✅ Successfully listed tables and their heads in: {out_file}")

✅ Successfully listed tables and their heads in: C:\Users\lazyst\Files\ucl\Dissertation\build\output\duckdb_tables.md


### post-hoc derived table merging on industry_codes

This script natively groups the boolean flags, isolates the unique string codes, concatenates them with commas, joins them back together, and overwrites the table with the pristine data.